# AI vs Real Face Detector — Colab Training

**Enable GPU:** Runtime -> Change runtime type -> T4 GPU

Do NOT run full training on an 8GB RAM laptop without GPU.


In [ ]:
# GPU check
import torch
assert torch.cuda.is_available(), 'Enable GPU runtime in Colab!'
print('GPU:', torch.cuda.get_device_name(0))


GPU: Tesla T4


In [ ]:
# Clone ONLY the master branch
%cd /content
!rm -rf /content/ai-vs-real-face-detector
!git clone --branch master --single-branch https://github.com/Algorithm-bot/ai-vs-real-face-detector.git /content/ai-vs-real-face-detector
!find /content -iname "train.py" 2>/dev/null

/content
Cloning into '/content/ai-vs-real-face-detector'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 132 (delta 34), reused 126 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 602.27 KiB | 2.21 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/ai-vs-real-face-detector/ai-vs-real-face-detector/src/train.py


## Mount Drive and point paths at persistent storage
Both the dataset (from `dataset_prep_colab.ipynb`) and the model checkpoints live on Drive, not the
local Colab disk -- `/content` is wiped if your session disconnects or times out mid-training, and you
do not want to lose a completed training run because of that.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_DIR = '/content/drive/MyDrive/ai-vs-real-face-detector/data'
OUTPUT_DIR = '/content/drive/MyDrive/ai-vs-real-face-detector/models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.isdir(os.path.join(DATA_DIR, 'real')), f"Missing {DATA_DIR}/real -- run dataset_prep_colab.ipynb first"
assert os.path.isdir(os.path.join(DATA_DIR, 'fake')), f"Missing {DATA_DIR}/fake -- run dataset_prep_colab.ipynb first"

print("Data dir:  ", DATA_DIR)
print("Output dir:", OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data dir:   /content/drive/MyDrive/ai-vs-real-face-detector/data
Output dir: /content/drive/MyDrive/ai-vs-real-face-detector/models


## Stage 1 -- Deep branch baseline
Watch the first printed lines for the "Train split composition" / "Val split composition" output --
confirm `real` and `fake/stylegan2` (and later `fake/diffusion`) counts look right before letting a
full run continue. Stop and investigate if any group shows 0.

Uses the `DATA_DIR` / `OUTPUT_DIR` Python variables set above -- IPython substitutes `{DATA_DIR}` and
`{OUTPUT_DIR}` directly into the shell command.


In [ ]:
!python /content/ai-vs-real-face-detector/ai-vs-real-face-detector/src/train.py --mode stage1 --data-dir "{DATA_DIR}" --output-dir "{OUTPUT_DIR}" --epochs 10 --batch-size 32 --num-workers 2

GPU INFORMATION
GPU: Tesla T4
CUDA: 12.8


TRAINING CONFIGURATION
Mode: stage1
Data: /content/drive/MyDrive/ai-vs-real-face-detector/data
Output: /content/drive/MyDrive/ai-vs-real-face-detector/models
Backbone: efficientnet_b0
Epochs: 10
Batch size: 32
Learning rate: 0.0001
Validation ratio: 0.15
Workers: 2
Seed: 42


STAGE 1 — DEEP BRANCH TRAINING

Total dataset composition:
  real/real: 5000
  fake/stylegan2: 3312
  TOTAL: 8312

TRAIN split:
  real/real: 4250
  fake/stylegan2: 2815
  TOTAL: 7065

Total dataset composition:
  real/real: 5000
  fake/stylegan2: 3312
  TOTAL: 8312

VAL split:
  real/real: 750
  fake/stylegan2: 497
  TOTAL: 1247

Training batches: 221
Validation batches: 39

[Stage1 Epoch 1/10] train loss=0.0572 acc=0.9885 | val loss=0.0013 acc=1.0000
Saved checkpoint: /content/drive/MyDrive/ai-vs-real-face-detector/models/stage1_best.pt

[Stage1 Epoch 2/10] train loss=0.0025 acc=0.9997 | val loss=0.0005 acc=1.0000

[Stage1 Epoch 3/10] train loss=0.0010 acc=1.0000 | val l

## Stage 3 -- Hybrid fusion training

> Add blockquote



> [Add blockquote](https://)


Warm-starts from `stage1_best.pt` in `OUTPUT_DIR` automatically if it exists (see `run_hybrid()` in
`train.py`), so run this only after Stage 1 above has completed and saved a checkpoint.



--- 13179.png ---
  imread+cvtColor: 0.003s
  physics extract: 1.989s
  preprocess_pil:  0.005s
  transform:       0.008s

--- 13188.png ---
  imread+cvtColor: 0.005s
  physics extract: 0.025s
  preprocess_pil:  0.000s
  transform:       0.008s

--- 13190.png ---
  imread+cvtColor: 0.005s
  physics extract: 0.032s
  preprocess_pil:  0.001s
  transform:       0.006s

--- 13192.png ---
  imread+cvtColor: 0.003s
  physics extract: 0.020s
  preprocess_pil:  0.001s
  transform:       0.004s

--- 13193.png ---
  imread+cvtColor: 0.003s
  physics extract: 0.040s
  preprocess_pil:  0.001s
  transform:       0.011s

All 5 images completed successfully.


In [ ]:
!python /content/ai-vs-real-face-detector/ai-vs-real-face-detector/src/train.py --mode hybrid --data-dir /content/data --output-dir "{OUTPUT_DIR}" --epochs 15 --batch-size 32 --num-workers 2

GPU INFORMATION
GPU: Tesla T4
CUDA: 12.8


TRAINING CONFIGURATION
Mode: hybrid
Data: /content/data
Output: /content/drive/MyDrive/ai-vs-real-face-detector/models
Backbone: efficientnet_b0
Epochs: 15
Batch size: 32
Learning rate: 0.0001
Validation ratio: 0.15
Workers: 0
Seed: 42


HYBRID TRAINING — DEEP + PHYSICS

Total dataset composition:
  real/real: 5000
  fake/stylegan2: 3312
  TOTAL: 8312

TRAIN split:
  real/real: 4250
  fake/stylegan2: 2815
  TOTAL: 7065

Total dataset composition:
  real/real: 5000
  fake/stylegan2: 3312
  TOTAL: 8312

VAL split:
  real/real: 750
  fake/stylegan2: 497
  TOTAL: 1247

Training batches: 221
Validation batches: 39

Loading Stage 1 checkpoint...
Loaded Stage 1 backbone weights.
Training:  15% 34/221 [01:01<05:19,  1.71s/it, acc=0.903, loss=0.126]/usr/local/lib/python3.12/dist-packages/skimage/measure/fit.py:530: RuntimeWarning: invalid value encountered in scalar divide
  phi = 0.5 * np.arctan((2.0 * b) / (a - c))
Traceback (most recent call last):


## Checkpoints are already on Drive
Because `--output-dir` points at `/content/drive/...`, `hybrid_best.pt` and `stage1_best.pt` are
already saved persistently -- no separate download step needed. Just sync Drive to your laptop, or
download the specific file below if you want it locally right away for `inference.py`.


In [ ]:
from google.colab import files
files.download(os.path.join(OUTPUT_DIR, 'hybrid_best.pt'))
